In [3]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-29 16:44:32--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-09-29 16:44:32 (157 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [4]:
with open ('/content/input.txt', 'r',encoding='utf-8') as f:
    # carregando o texto de entrada shakespeare
    text = f.read()

In [5]:
print("llen characters: ",len(text))
# quanntidade de caracteres no texto

llen characters:  1115394


In [6]:
print(text[:1000])
# 1000 priemiros carateres

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [7]:
chars = sorted(list(set(text)))
# criando uma lista ordenada somentte com os caracteres unicos no texto
vocab_size = len((chars))
# tamanho do vacabulario
print(''.join(chars))
print(vocab_size)
# o vocabulario é o conjunto de caracteres possiveis dentro do texto


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
stoi = {ch:i for i,ch in enumerate(chars)}
# para cada caractere inumera cada um com um numero inteiro(como um indice)
itos = {i:ch for i,ch in enumerate(chars)}
# mesmo mapeameto mas agora invertido
encode = lambda s: [stoi[c] for c in s]
#dado um astring pega o caracctere em stoi, retornando uma sequencia de numeros
decode = lambda l: ''.join([itos[i] for i in l])
#recebendo uma lista de inteiros, forma uma frase com os carcteres mapeados

print(encode("hi there"))
print(decode(encode("hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [9]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)
# tokenizando todo o texto do shakespeare e ttransformando-o em um tensor
print(data.shape, data.dtype)
#tamnho do tensor e tipo
print(text[:1000])

torch.Size([1115394]) torch.int64
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for b

In [10]:
n = int(0.9*len(data))
# separando 90% da base de shakespeare
train_data = data[:n]
# 90% dos dados vao ficar para o treino
val_data = data[n:]
# os outros 10% ficam para a validação da rede

In [11]:
block_size = 8
# o treinamento de um transformer é feito em blocos por isso é definido o tamanho do bloco que vai para o treinamento
train_data[:block_size+1]
# pegando o bloco na base de treino

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
x = train_data[:block_size]
# pegando  o bloco na base de treino
y=train_data[1:block_size+1]
# pega o mesmo bloco porem um idice a frente, ser ao que o modelo vai tentar prever
for t in range(block_size):
    #para cada indice no bloco
    #pega o indice +1 no bloco de indices, crescendo gradativamente
    context = x[:t+1]
    # e pega o indice posterior como target para dizer como o previsto
    target = y[t]
    print(f'quando a entrada é {context} a saida é: {target}')

quando a entrada é tensor([18]) a saida é: 47
quando a entrada é tensor([18, 47]) a saida é: 56
quando a entrada é tensor([18, 47, 56]) a saida é: 57
quando a entrada é tensor([18, 47, 56, 57]) a saida é: 58
quando a entrada é tensor([18, 47, 56, 57, 58]) a saida é: 1
quando a entrada é tensor([18, 47, 56, 57, 58,  1]) a saida é: 15
quando a entrada é tensor([18, 47, 56, 57, 58,  1, 15]) a saida é: 47
quando a entrada é tensor([18, 47, 56, 57, 58,  1, 15, 47]) a saida é: 58


In [13]:
torch.manual_seed(1337)
# define uma seed
batch_size = 4
# quantos batches serao processsados simultaneamente
block_size = 8
# tamanho do bloco

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # esolhe o tensor de treino se split for treino, caso nao pega o val
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # pega indices iniciais aleatórios entre 0 e o tamanho da base - o tamanho do bloco
    x = torch.stack([data[i:i+block_size] for i in ix])
    # para cada iteração nos indices aleatórios, pega um aparte do block em caracteres que começa em i
    # o torch.stack coloca isso em um tensor 2d
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    # para cada i pega o proximo indice depois de um incie em x
    return x, y

xb, yb = get_batch('train')
# gerando um batch
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):
    for t in range(block_size): # para cada indice em um range do tamanho do block, pega o contexto e sua previsao
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [14]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    # criando uma classe que herda o nn.module padrao pytorch para rede
    def __init__(self,vocab_size):
        # vocabulario do texto
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
        # cria uma "matriz", onde para cada indice de caractere cria um vetor representando aquele indice

    def forward(self, idx,targets=None):
        #recebe os indices de treino
        logits = self.token_embedding_table(idx)
        # proximos valores de caracteres normalizados (previsao dos proximos caracteres com abase em um unico token)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            #pegando cada dimensão
            logits = logits.view(B*T,C)
            # a funcao de loss espera um tensor 2D
            targets = targets.view(B*T)
            # transforma em um vetor unidimensional
            loss = F.cross_entropy(logits,targets)
            #calcula o loss
        return logits, loss

    def generate(self,idx,max_new_tokens):
        # recebe o array de contexto(idx) e o numero maximo de tokens para gerar
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # pred e loss para cada indice
            logits = logits[:,-1,:]
            # pega o utlimo caractere e preve em cima dele
            probs = F.softmax(logits,dim=-1)
            # aplica o softmax a ultima dimensao assim retornando uma sistribuição de probabilidade
            idx_next = torch.multinomial(probs, num_samples=1)
            # pega uma amostra da distribuição de probs
            idx=torch.cat((idx,idx_next),dim=1)
            # coloca o token ou sequncia de token ao final do contexto
        return idx
    # retorna o contexto mais o gerado

m = BigramLanguageModel(vocab_size)
# instanciando o modelo
logits, loss = m(xb,yb)
# passando os indices para o modelo
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))
# gera 100 novos tokens a partir do token de entrada, retornando uma lista com a sequencia de tokens

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [16]:
optimizer = torch.optim.AdamW(m.parameters(), lr=0.001)
# criando o optimizer que sera utilizado para aplicar a mudação de pesos a rede

In [17]:
batch_size = 32
# tamanho do lote que ira para rede
for steps in range(10000):
    # 10000 epocas de treino

    xb, yb = get_batch('train')
    # pegando os dados de treino e target de treino

    logits, loss = m(xb, yb)
    # pegando os logists e loss
    optimizer.zero_grad(set_to_none=True)
    # zera o gradiente
    loss.backward()
    #  backpropagation (otimmizador volta pela rede aoplicando os pesos)
    optimizer.step()

print(loss.item())

2.382369041442871


In [18]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))
# gerando 500 tokens


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


In [19]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [20]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # quantas sequencias | tamanho do bloco | canais(dimensao do vetor)
x = torch.randn(B,T,C)
# cria um tensor de 3 dimensoes onde tem numeros de um distribuição 0-1
x.shape

torch.Size([4, 8, 2])

In [21]:
xbow = torch.zeros((B,T,C))
# tensor 3d preenchidos com 0
for b in range(B):
    # para cada b em block_size
    for t in range(T):
        #preve a sequencia para o tamanho de sequencias(8)
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)
        # media

In [22]:
wei = torch.tril(torch.ones(T, T))
# faz uma matriz, onde todos os valores acima da diagonal pricipal são zerados, ou seja é ate onde o modelo cosegue ver os proximos tokens dado seu indice
wei = wei / wei.sum(1, keepdim=True)
# soma todos os numeros da linha e divide pela matriz wei
xbow2 = wei @ x
# faz a media dos vetores ate os tokens vistos
torch.allclose(xbow, xbow2)
# comparando as matrizes a original e a normalizada, verificando se sao equivalentes

False

In [23]:
# chegando ao mesmo resultado obtido em xbow2 porem agora sem precisar fazer a divisao na mao
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
# apllicando o softmax
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [24]:
torch.manual_seed(1337)
B,T,C = 4,8,32
# definindo as dimensoes
x = torch.randn(B,T,C)
# transformando em um tensor

head_size = 16
# quanto uma posição deve olhar para outra
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
wei =  q @ k.transpose(-2, -1)
# para cada batch em B cria-se uma matriz de afinidades para os tokens em cada sequencia em B

# cada token so pode enxergar o ele mesmo e o que vem antes dele, chamada de mascara
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
# normalizando

v = value(x)
out = wei @ v
# representação de cada valor em determinada posição, dado o que ja foi visto

out.shape

torch.Size([4, 8, 16])

In [25]:
wei[0]
# a afinidade de dois tokens se define por quanto um token 4 por exemplo, na previsao de numeros posteriores a 8, onde a informação de 8 tem uma afinidade alta para 4
# por exemplo, a plavra "O cachorro pintado esta correndo" o token 2 influencia quanto o token 4(esta) tem peso alto para o token 2

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [26]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5
# teste com k e q aleatorios, divide pela raizz de head para coloca a variancia proximo de 1

In [27]:
k.var()
# variancia de k

tensor(1.0449)

In [28]:
q.var()
#variancia q

tensor(1.0700)

In [29]:
wei.var()

tensor(1.0918)

In [30]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [31]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1)

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [32]:
class LayerNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # garante que nao aconteça divisao por 0 e que o modelo possa desnormalizar

  def __call__(self, x):
    xmean = x.mean(1, keepdim=True)
    xvar = x.var(1, keepdim=True)
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
    self.out = self.gamma * xhat + self.beta
    return self.out
  # para para cada x é calculado a media e a variancia das features

  def parameters(self):
    return [self.gamma, self.beta]
  # mostra o sparametros

torch.manual_seed(1337)
module = LayerNorm1d(100)
# chama a camada de normalização com 100 dimensoes
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [33]:
x[:,0].mean(), x[:,0].std()

(tensor(0.1469), tensor(0.8803))

In [34]:
x[0,:].mean(), x[0,:].std()

(tensor(-9.5367e-09), tensor(1.0000))

In [35]:
import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5091, val loss 2.5058
step 300: train loss 2.4197, val loss 2.4336
step 400: train loss 2.3501, val loss 2.3562
step 500: train loss 2.2963, val loss 2.3125
step 600: train loss 2.2407, val loss 2.2496
step 700: train loss 2.2054, val loss 2.2187
step 800: train loss 2.1633, val loss 2.1866
step 900: train loss 2.1241, val loss 2.1504
step 1000: train loss 2.1036, val loss 2.1306
step 1100: train loss 2.0698, val loss 2.1180
step 1200: train loss 2.0380, val loss 2.0791
step 1300: train loss 2.0248, val loss 2.0634
step 1400: train loss 1.9926, val loss 2.0359
step 1500: train loss 1.9697, val loss 2.0287
step 1600: train loss 1.9627, val loss 2.0477
step 1700: train loss 1.9403, val loss 2.0115
step 1800: train loss 1.9090, val loss 1.9941
step 1900: train loss 1.9092, val loss 1.9858
step 2000: train loss 1.8847, val loss 1.9925
step 2100: train loss 1.